In [4]:
print("hello")

hello


In [5]:
%pwd


'c:\\Users\\hp\\OneDrive\\Desktop\\medical-chatbot\\research'

In [6]:
import os
os.chdir("C:/Users/hp/OneDrive/Desktop/medical-chatbot")


In [7]:
%pwd

'C:\\Users\\hp\\OneDrive\\Desktop\\medical-chatbot'

In [8]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

C:\Users\hp\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# Extract text from PDF files
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents

In [10]:
extracted_data = load_pdf_files("data")

In [11]:
extracted_data

[Document(metadata={'producer': 'Adobe PDF Library 9.0', 'creator': 'Acrobat 9.3.3', 'creationdate': '2010-07-24T16:57:37-07:00', 'moddate': '2010-07-28T15:49:40-07:00', 'trapped': '/False', 'source': 'data\\14.DavidWerner-WhereThereIsNoDoctor.pdf', 'total_pages': 503, 'page': 0, 'page_label': 'i'}, page_content='Where There Is No Doctor 2010'),
 Document(metadata={'producer': 'Adobe PDF Library 9.0', 'creator': 'Acrobat 9.3.3', 'creationdate': '2010-07-24T16:57:37-07:00', 'moddate': '2010-07-28T15:49:40-07:00', 'trapped': '/False', 'source': 'data\\14.DavidWerner-WhereThereIsNoDoctor.pdf', 'total_pages': 503, 'page': 1, 'page_label': 'ii'}, page_content='Where There Is No Doctor 2010\nLibrary of Congress Cataloging-in-Publication Data\nThe Library of Congress has already cataloged the 10-digit ISBN as follows: \nWerner, David, 1934-\n    Where there is no doctor: a village health care handbook / by David Werner; \nwith Carol Thuman and Jane Maxwell-Rev. ed.\nIncludes Index.\nISBN 0-94

In [12]:
len(extracted_data)

1140

In [13]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [14]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [15]:
minimal_docs

[Document(metadata={'source': 'data\\14.DavidWerner-WhereThereIsNoDoctor.pdf'}, page_content='Where There Is No Doctor 2010'),
 Document(metadata={'source': 'data\\14.DavidWerner-WhereThereIsNoDoctor.pdf'}, page_content='Where There Is No Doctor 2010\nLibrary of Congress Cataloging-in-Publication Data\nThe Library of Congress has already cataloged the 10-digit ISBN as follows: \nWerner, David, 1934-\n    Where there is no doctor: a village health care handbook / by David Werner; \nwith Carol Thuman and Jane Maxwell-Rev. ed.\nIncludes Index.\nISBN 0-942364-15-5\n1. Medicine, Popular. 2. Rural health. I. Thuman, Carol,\n1959-. II. Maxwell, Jane, 1941-. III Title.\n[DNLM: 1. Community Health Aides-handbooks.\n2. Medicine-popular works. 3. Rural Health-handbooks.\nWA 39 W492W]\nRC81.W4813 1992 610-dc20\nDNLM/DLC\n \n  92-1539\nfor Library of Congr\ness CIP\nTHIS REVISED EDITION CAN BE IMPROVED WITH  YOUR HELP.  \nIf you are a community health worker, doctor, mother, or anyone with ideas or

In [16]:
# Split the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [17]:
texts_chunk = text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunk)}")

Number of chunks: 8375


In [18]:
texts_chunk

[Document(metadata={'source': 'data\\14.DavidWerner-WhereThereIsNoDoctor.pdf'}, page_content='Where There Is No Doctor 2010'),
 Document(metadata={'source': 'data\\14.DavidWerner-WhereThereIsNoDoctor.pdf'}, page_content='Where There Is No Doctor 2010\nLibrary of Congress Cataloging-in-Publication Data\nThe Library of Congress has already cataloged the 10-digit ISBN as follows: \nWerner, David, 1934-\n    Where there is no doctor: a village health care handbook / by David Werner; \nwith Carol Thuman and Jane Maxwell-Rev. ed.\nIncludes Index.\nISBN 0-942364-15-5\n1. Medicine, Popular. 2. Rural health. I. Thuman, Carol,\n1959-. II. Maxwell, Jane, 1941-. III Title.\n[DNLM: 1. Community Health Aides-handbooks.'),
 Document(metadata={'source': 'data\\14.DavidWerner-WhereThereIsNoDoctor.pdf'}, page_content='2. Medicine-popular works. 3. Rural Health-handbooks.\nWA 39 W492W]\nRC81.W4813 1992 610-dc20\nDNLM/DLC\n \n  92-1539\nfor Library of Congr\ness CIP\nTHIS REVISED EDITION CAN BE IMPROVED W

In [19]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return the HuggingFace embeddings model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings

embedding = download_embeddings()

C:\Users\hp\AppData\Local\Temp\ipykernel_26896\30168171.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [20]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [21]:
vector = embedding.embed_query("Hello world")
vector

[-0.034477248787879944,
 0.031023213639855385,
 0.006734968163073063,
 0.02610895223915577,
 -0.03936203941702843,
 -0.16030244529247284,
 0.06692398339509964,
 -0.006441461853682995,
 -0.0474504716694355,
 0.014758830890059471,
 0.07087528705596924,
 0.05552760139107704,
 0.019193364307284355,
 -0.026251312345266342,
 -0.010109573602676392,
 -0.026940463110804558,
 0.022307418286800385,
 -0.02222663164138794,
 -0.149692565202713,
 -0.017493005841970444,
 0.007676235865801573,
 0.05435224995017052,
 0.003254480427131057,
 0.0317259207367897,
 -0.0846213772892952,
 -0.02940598502755165,
 0.051595576107501984,
 0.048124026507139206,
 -0.0033147847279906273,
 -0.05827918276190758,
 0.04196928068995476,
 0.02221066877245903,
 0.1281888484954834,
 -0.02233896031975746,
 -0.011656244285404682,
 0.06292830407619476,
 -0.03287629783153534,
 -0.09122602641582489,
 -0.03117540292441845,
 0.05269957706332207,
 0.04703482240438461,
 -0.0842030718922615,
 -0.03005615808069706,
 -0.02074484899640083

In [22]:
print( "Vector length:", len(vector))

Vector length: 384


In [23]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [24]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [25]:
from pinecone import Pinecone 
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [26]:
pc

In [27]:
from pinecone import ServerlessSpec 

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384,  # Dimension of the embeddings
        metric= "cosine",  # Cosine similarity
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )


index = pc.Index(index_name)

In [28]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

In [29]:
# Load Existing index 

from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

# Add more data in to pinecone index


In [30]:
dswith = Document(
    page_content="Aayush chauhan is a b.tech computer science student from graphic era hill university.",
    metadata={"source": "Aayush"}
)

In [31]:
docsearch.add_documents(documents=[dswith])

['bc29ab5a-c753-4995-8d63-403f033dd0af']

In [32]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [33]:
retrieved_docs = retriever.invoke("What is Acne?")
retrieved_docs

[Document(id='07323e22-d0ed-431f-84ed-0907f8af75fa', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='e353c039-914e-4498-b396-144b38c6ae63', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='8cfbff27-d302-420c-a3dd-72214c95ebdb', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26')]

In [34]:
import os
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(
    model="openai/gpt-oss-120b",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENAI_API_KEY")
)


In [35]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [36]:
system_prompt = (
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [37]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [38]:
response = rag_chain.invoke({"input": "what is Acromegaly and gigantism?"})
print(response["answer"])

Acromegaly is a disorder caused by excess growth‑hormone (GH) secretion from a pituitary tumor after the growth plates have closed, leading to abnormal enlargement of bone and soft tissue and various systemic effects. Gigantism results from the same GH excess but occurs before the epiphyses fuse, producing excessive linear height growth. Both conditions stem from uncontrolled pituitary hormone release.


In [39]:
response = rag_chain.invoke({"input": "what  is Aayush chauhan?"})
print(response["answer"])

Aayush Chauhan is a B.Tech computer science student attending Graphic Era Hill University.
